In [ ]:
# Loading stuff

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
import uuid
%matplotlib inline

## Data exploration and analysis

Here we:
- Loaded the datasets
- Dropped all object type colmns composed only by unique values
- Dropped all columns composed by the same value for all rows
- Encoded the target variable
- Oversampled the minority transition class
- Balanced the dataset by picking equal number of samples from each class (NOT SURE IF WORTH IT, NEEDS TO BE TESTED)
- Dropped non-target columns with high correlation between them
- Droppped columns with low correlation with the target variable (NOT SURE IF WORTH IT, NEEDS TO BE TESTED)
- Treated outlier values (NOT SURE IF WORTH IT, NEED TO TEST DIFFERENT VALUES)


In [ ]:
# --- Loading all the datasets
# Train dataset
train_dataset = pd.read_csv('data/train_radiomics_hipocamp.csv')
# Test dataset
test_dataset = pd.read_csv('data/test_radiomics_hipocamp.csv')

In [ ]:
# --- Exploring the train dataset
train_dataset.head()
train_dataset.columns
train_dataset.info(verbose=True, show_counts=True)

In [ ]:
# Getting all columns whose value is an object
obj_col = train_dataset.select_dtypes(include='object').columns

# For each of those, checking the number of unique values
# Storing in a list the columns that have the same number of unique values as the number of rows
# This means that the column is a unique identifier and probably should be dropped
unique_cols = []
for col in obj_col:
    if train_dataset[col].nunique() == 305:
        unique_cols.append(col)

# Creating and displaying a dataframe containing only the columns stored in the above list
unique_df = train_dataset[unique_cols]

# It looks like all the columns in the unique_df dataframe are unique identifiers that don't provide any useful information

# Dropping the columns stored in the unique_cols list
train_dataset.drop(unique_cols, axis=1, inplace=True)
# Dropping them from the test dataset as well
test_dataset.drop(unique_cols, axis=1, inplace=True)

In [ ]:
# Dropping all collumns with only one unique value

only_one_unique_val_col = 0
for col in train_dataset.columns:
    if train_dataset[col].nunique() == 1:
        only_one_unique_val_col += 1
        train_dataset.drop(col, axis=1, inplace=True)
        # Dropping them from the test dataset as well
        test_dataset.drop(col, axis=1, inplace=True)

print("dropped " + str(only_one_unique_val_col) + " columns")
        

In [ ]:
# Checking all unique values in the Transition column
train_dataset['Transition'].value_counts()
# Dataset is extremely UNBALANCED

In [ ]:
# --- Using encoding to transform the categorical columns into numerical columns
replace_map = {'Transition': {'CN-CN': 0, 'AD-AD': 1, 'CN-MCI': 2, 'MCI-AD': 3, 'MCI-MCI': 4}}
train_dataset.replace(replace_map, inplace=True)
test_dataset.replace(replace_map, inplace=True)
train_dataset.head()

In [ ]:
train_dataset['Transition'].value_counts()

In [ ]:
train_dataset.info()

In [ ]:
# --- Droppig non-target columns with the same info (correlation = 1 between them)

#target_column = 'Transition'  
## Calculating the correlation matrix (excluding the target column and the ID column)
#train_dataset_tmp = train_dataset 
#corr_matrix_train = train_dataset_tmp.drop(columns=[target_column]).corr().abs()
#
## Upper triangle matrix of correlations
#upper = corr_matrix_train.where(np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool))
#
## Finding index of feature columns with correlation equal to 1
#threshold = 1.0
#to_drop = [column for column in upper.columns if any(upper[column] >= threshold)] # for some reason there are columns with correlation > 1.0
#print("dropping " + str(len(to_drop)) + " columns")
#
## Dropping the columns
#train_dataset = train_dataset.drop(columns=to_drop)
#test_dataset = test_dataset.drop(columns=to_drop)

In [ ]:
# Checking dataset info
train_dataset.info()

In [ ]:
## --- Dropping columns with low correlation with the target column (<0.005)

#corr_matrix = train_dataset.corr()
#corr_target = corr_matrix['Transition'].abs()
#uncorr_cols = []
#for i in range(len(corr_target)):
#    if corr_target[i] < 0.005:
#        uncorr_cols.append(corr_matrix.columns[i])
#
## Dropping the columns stored in the uncorr_cols list
#n_cols_before = len(train_dataset.columns)
#train_dataset.drop(uncorr_cols, axis=1, inplace=True)
#test_dataset.drop(uncorr_cols, axis=1, inplace=True)
#n_cols_after = len(train_dataset.columns)
#
#print("Dropped " + str(n_cols_before-n_cols_after) + " columns!")

In [ ]:
# Checking Age column
print(train_dataset['Age'].hist())
plt.show()
# Not a normal distribution

In [ ]:
# Checking Sex column
print(train_dataset['Sex'].value_counts(normalize=True))
# Not quite balanced

In [ ]:
# Understanding the relationship between Age and Transition
sns.catplot(x="Transition", y="Age", data=train_dataset, kind="box", aspect=1.5)
plt.title("Boxplot for Transition vs Age")
plt.show()

In [ ]:
train_dataset.head()

In [ ]:
# Treating outliers in all columns
# Outliers should be converted to the closest bound

changes_count = 0

for col in train_dataset.columns:
    if col not in ['Transition', 'ID']:
        median = train_dataset[col].median()
        q1 = train_dataset[col].quantile(0.25)
        q3 = train_dataset[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # Count outliers before replacing them
        changes_count += ((train_dataset[col] < lower_bound) | (train_dataset[col] > upper_bound)).sum()
        changes_count += ((test_dataset[col] < lower_bound) | (test_dataset[col] > upper_bound)).sum()
        
        # Replace outliers with the median
        # train_dataset[col] = train_dataset[col].apply(lambda x: median if x < lower_bound or x > upper_bound else x)
        # test_dataset[col] = test_dataset[col].apply(lambda x: median if x < lower_bound or x > upper_bound else x)

        # Replace outliers with the closest bound
        train_dataset[col] = train_dataset[col].apply(lambda x: lower_bound if x < lower_bound else upper_bound if x > upper_bound else x)
        test_dataset[col] = test_dataset[col].apply(lambda x: lower_bound if x < lower_bound else upper_bound if x > upper_bound else x)

print(f"Total number of changes made: {changes_count}")


In [ ]:
train_dataset['Transition'].value_counts()

In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd

# Add an 'oversampled' column to the dataset with 'NO' values
train_dataset['oversampled'] = 'NO'

# Split the dataset into features and target
X = train_dataset.drop(columns=['Transition', 'oversampled'])
y = train_dataset['Transition']

# Identify the majority class count
majority_class_count = y.value_counts().max()

# Define a sampling strategy to oversample each class to match the majority class count
sampling_strategy = {
    cls: majority_class_count for cls, count in y.value_counts().items() if count < majority_class_count
}

# Apply SMOTE with the defined strategy
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

# Create a new dataframe with the oversampled data
oversampled_data = pd.concat([X_smote, y_smote], axis=1)
oversampled_data.columns = list(X.columns) + ['Transition']

# Mark the oversampled rows with 'YES' in the 'oversampled' column
oversampled_data['oversampled'] = ['YES' if idx >= len(X) else 'NO' for idx in range(len(X_smote))]

# Concatenate the new oversampled data with the original data
train_dataset = pd.concat([train_dataset, oversampled_data[oversampled_data['oversampled'] == 'YES']], ignore_index=True)

# Display the final class distribution for verification
print("Class distribution after SMOTE oversampling:")
print(train_dataset['Transition'].value_counts())

In [ ]:
# get number of normal and oversampled data
train_dataset['oversampled'].value_counts()


In [ ]:
# Normalizing the dataset (excluding the target column) form 0 to 1
#from sklearn.preprocessing import MinMaxScaler
#
#scaler = MinMaxScaler()
## Exclude the target column 'Transition' from scaling
#train_features = train_dataset.drop(['Transition', 'oversampled'], axis=1)
#
## Scale the features
#train_dataset[train_features.columns] = scaler.fit_transform(train_features)
#test_dataset[test_dataset.columns] = scaler.transform(test_dataset)

In [ ]:
# Saving the treated datasets
train_dataset.to_csv('data/train_dataset.csv', index=False)
test_dataset.to_csv('data/test_dataset.csv', index=False)